# Deep Learning

Este notebook tem como objetivo documentar e explicar a utilização de Deep Learning no projeto, com foco na geração de características tabulares e na contribuição dessas características para o classificador multimodal responsável pela classificação dos chamados de suporte de TI.

Diferentemente de uma abordagem que utiliza apenas os dados originais dos chamados, o projeto busca combinar informações de diferentes modalidades, incluindo texto, imagens e variáveis estruturadas. Nesse contexto, o Deep Learning é utilizado para aprender representações dos dados tabulares que possam complementar as informações obtidas pelas demais etapas do sistema.

## Contextualização

A arquitetura implementada utiliza um **autoencoder tabular**, uma rede neural treinada para reconstruir suas próprias entradas. Durante esse processo, a rede aprende uma representação compacta dos dados, denominada representação latente.

Em seguida, essas representações são agrupadas por meio do algoritmo K-Means, gerando perfis de chamados com características semelhantes. O resultado é disponibilizado na coluna `tabular_profile_cluster`, que pode ser utilizada como uma das características do classificador multimodal.

É importante distinguir as duas etapas: o autoencoder não realiza diretamente a classificação final do chamado. Sua função é aprender representações dos dados e contribuir para a geração de características; a previsão de `target_category` é responsabilidade do classificador multimodal.

## Escopo da documentação

Ao longo deste notebook, serão apresentados os fundamentos das redes neurais, o funcionamento do autoencoder, o processo de treinamento, a geração dos perfis tabulares e a integração dessa característica com as demais modalidades do projeto.

Também serão discutidos os cuidados necessários para evitar vazamento de dados, preservar a separação entre treinamento, validação e teste e avaliar se as características geradas realmente contribuem para a classificação final.

O foco será compreender **como o Deep Learning funciona dentro da arquitetura do projeto e qual é o papel de cada componente**, relacionando os conceitos teóricos às implementações existentes no código-fonte.

## 1) Arquitetura: do Deep Learning tabular ao classificador multimodal
Nesta seção, vamos acompanhar o caminho percorrido pelos dados até a classificação do chamado. O objetivo é entender onde o autoencoder se encaixa, o que ele entrega e como essa saída pode contribuir para a decisão do classificador multimodal.

### O papel de cada componente
1. **Autoencoder tabular**

O autoencoder recebe as variáveis estruturadas dos chamados e aprende a representá-las em um espaço de menor dimensão. Essa representação pode concentrar padrões presentes nos dados sem precisar utilizar diretamente a categoria que queremos prever.

2. **K-Means e geração de perfis**

Após a representação latente ser obtida, o K-Means agrupa registros com representações semelhantes. O grupo atribuído a cada chamado é armazenado na característica tabular_profile_cluster.

É importante destacar que os identificadores desses grupos são resultados de agrupamento, e não as categorias reais dos chamados. Por exemplo, o perfil 0 não significa necessariamente Software, assim como o perfil 1 não significa necessariamente Rede.

3. **Extração de características de texto e imagem**

O projeto também contempla componentes próprios para obter informações de outras modalidades. Cada componente transforma sua entrada em características que podem ser utilizadas na arquitetura multimodal.

4. **Fusão e classificação**

As características geradas são combinadas pelo fluxo multimodal. A MLP (Multilayer Perceptron) atua como classificador supervisionado: durante o treinamento, aprende relações entre as características combinadas e a categoria conhecida de cada chamado.

### Por que separar essas etapas?
Essa arquitetura divide um problema complexo em tarefas menores e especializadas. O autoencoder aprende uma representação dos dados tabulares, o K-Means organiza essas representações em grupos e o classificador multimodal aprende a relacionar as informações combinadas às categorias dos chamados.

Essa separação também permite avaliar as etapas individualmente e investigar se a inclusão dos perfis tabulares realmente contribui para a classificação final.

### O que precisamos verificar no código
Ao longo das próximas seções, vamos aprofundar os componentes que tornam esse fluxo possível:
- Como as variáveis tabulares são preparadas para a rede neural.
- Como o autoencoder é construído e treinado.
- Como a representação latente é utilizada pelo K-Means.
- Como o perfil gerado chega ao conjunto de características multimodais.
- Como a MLP utiliza essas características para realizar a classificação supervisionada.
Esses pontos serão detalhados com base nas implementações e configurações do projeto, distinguindo o comportamento previsto pelo código dos resultados que precisariam ser comprovados por uma execução.

## 3) Fundamentos de Deep Learning
Antes de analisar o autoencoder utilizado no projeto, precisamos entender os conceitos que permitem a uma rede neural aprender padrões nos dados.
### O que é Deep Learning?
Deep Learning, ou aprendizado profundo, é uma área do Machine Learning que utiliza redes neurais artificiais compostas por camadas de operações matemáticas. Durante o treinamento, essas redes ajustam seus parâmetros para aprender relações presentes nos dados.

Uma rede neural recebe valores numéricos, processa esses valores por meio de suas camadas e produz uma saída. O treinamento ajusta os pesos e vieses dessas camadas para reduzir uma função de perda (loss function), que mede a diferença entre a saída produzida e o resultado esperado para a tarefa.

No projeto, o Deep Learning é aplicado à geração de representações dos dados tabulares dos chamados, e não diretamente à previsão da categoria final nessa etapa.
### Como funciona um neurônio artificial?
Um neurônio artificial combina suas entradas com pesos, adiciona um viés e aplica uma função de ativação.

Matematicamente, podemos representar essa operação por:

z = sum(w_i * x_i) + b (de i=1 até n)

a = f(z)

Em que:
- \(x_i\): valores recebidos como entrada.
- \(w_i\): pesos ajustados durante o treinamento.
- \(b\): viés (bias) do neurônio.
- \(f\): função de ativação.
- \(a\): saída do neurônio.

Os pesos determinam a contribuição de cada entrada, enquanto o viés permite deslocar a transformação matemática. A função de ativação introduz não linearidade, possibilitando aprender relações mais complexas do que uma simples transformação linear.

### Camadas de uma rede neural

Uma rede pode conter várias camadas ocultas e diferentes quantidades de neurônios. A profundidade, a largura e as funções utilizadas influenciam a capacidade de representação e o custo computacional.

No autoencoder do projeto, a rede é densa (fully connected): cada neurônio de uma camada está conectado aos neurônios da camada seguinte.

### Como a rede aprende?
O treinamento de uma rede neural envolve um ciclo de operações:
1. **Propagação direta (forward pass)**
   
   Os dados percorrem a rede e produzem uma saída.
2. **Cálculo da perda (loss)**
   
   A saída é comparada com o resultado esperado para a tarefa.
3. **Retropropagação (backpropagation)**
   
   A rede calcula gradientes que indicam como os parâmetros contribuem para o erro.
4. **Atualização dos parâmetros**
   
   O otimizador ajusta os pesos e vieses para tentar reduzir a perda.
5. **Repetição**
   
   O processo é repetido ao longo dos lotes e das épocas de treinamento.

No treinamento do autoencoder, o resultado esperado é a própria entrada. Isso significa que a rede aprende a reconstruir os dados que recebeu, em vez de receber diretamente a categoria do chamado como resposta correta.
### Conceitos importantes para o projeto
Conceito	Significado neste contexto
Época (epoch)	Uma passagem completa pelo conjunto de treinamento.
Lote (batch)	Subconjunto de registros processado antes de uma atualização dos parâmetros.
Função de perda	Mede o erro de reconstrução entre a entrada e a saída do autoencoder.
Otimizador	Atualiza os parâmetros da rede com base nos gradientes calculados.
Validação	Permite acompanhar a capacidade de reconstrução em dados separados do treinamento.
Early stopping	Interrompe o treinamento quando a perda de validação deixa de melhorar por determinado número de épocas.

No projeto, o autoencoder utiliza o otimizador Adam e o erro quadrático médio (Mean Squared Error — MSE) como função de perda. A configuração estabelece até 30 épocas, lotes de 64 registros e paciência de 5 épocas para o early stopping.

Esses valores representam os parâmetros configurados; não significam que o treinamento necessariamente executou todas as épocas.
### Relação com o autoencoder tabular
Os fundamentos apresentados explicam como o autoencoder consegue aprender uma representação dos chamados. Entretanto, existe uma particularidade: sua arquitetura precisa reconstruir as entradas enquanto aprende uma representação intermediária mais compacta.

Essa representação compacta é chamada de espaço latente. É nela que o projeto aplica o K-Means para agrupar chamados com padrões tabulares semelhantes.

## 4) Autoencoder tabular

O autoencoder é uma arquitetura de rede neural utilizada para aprender representações compactas dos dados. Diferentemente de um classificador tradicional, ele não precisa receber a categoria correta de cada registro durante o treinamento. Seu objetivo é aprender a reconstruir os próprios dados de entrada.

No Projeto Integrador II, o autoencoder é utilizado para processar variáveis estruturadas dos chamados de suporte de TI, como tempo de SLA, sentimento, quantidade de palavras, tipo de dispositivo, sistema operacional e tipo de rede.

### Estrutura da arquitetura

A implementação está localizada em `src/models/profiles.py`, na função `build_autoencoder()`.

A arquitetura possui três componentes principais:

- **Encoder:** recebe os dados pré-processados e os transforma em uma representação compacta.
- **Espaço latente:** armazena a representação reduzida aprendida pela rede neural.
- **Decoder:** utiliza a representação compacta para reconstruir os dados de entrada.

O fluxo de processamento pode ser representado da seguinte forma:

**Dados de entrada → Encoder → Representação latente → Decoder → Dados reconstruídos**

No projeto, a rede utiliza as seguintes dimensões configuradas:

- Dimensão da camada oculta (`hidden_dim`): 32 neurônios.
- Dimensão da representação latente (`latent_dim`): 8 neurônios.

A dimensão de entrada (`input_dim`) depende da quantidade de variáveis resultante do pré-processamento, incluindo as colunas categóricas transformadas em representações numéricas.

### Implementação em PyTorch

A função `build_autoencoder()` constrói o encoder e o decoder utilizando camadas `Linear` e a função de ativação `ReLU`.

```python
self.encoder = nn.Sequential(
    nn.Linear(input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim, latent_dim)
)

self.decoder = nn.Sequential(
    nn.Linear(latent_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim, input_dim)
)

def forward(self, values):
    return self.decoder(self.encoder(values))
```

A primeira camada do encoder recebe os dados e os transforma em uma representação intermediária de 32 dimensões. Em seguida, a rede produz a representação latente de 8 dimensões.

O decoder realiza o caminho inverso: recebe a representação latente, passa por uma camada oculta de 32 dimensões e produz uma saída com a mesma dimensão da entrada pré-processada.

O método `forward()` conecta essas etapas, primeiro executando o encoder e depois o decoder. Assim, durante o treinamento, a rede aprende a reconstruir os dados a partir de uma representação menor.

### O que é a representação latente?

A representação latente é um vetor numérico que resume características dos dados de entrada. Neste projeto, cada registro é representado por um vetor de 8 dimensões, aprendido pelo encoder.

Essa redução permite trabalhar com uma representação mais compacta das variáveis estruturadas. Entretanto, cada dimensão latente não corresponde necessariamente a uma característica específica e interpretável, como “urgência” ou “problema de rede”. A rede aprende combinações numéricas durante o treinamento.

É importante destacar que **a representação latente, e não a saída reconstruída pelo decoder, é utilizada na etapa seguinte para agrupar os registros**.

### Como o autoencoder aprende?

Durante o treinamento, o modelo compara os dados reconstruídos com os dados originais pré-processados. A diferença entre essas duas representações é calculada por meio da função de perda *Mean Squared Error* (MSE).

Em termos conceituais, o objetivo é minimizar:

\[
L = \frac{1}{N}\sum_{i=1}^{N}(x_i-\hat{x}_i)^2
\]

Em que:

- \(x_i\) representa um valor original de entrada;
- \(\hat{x}_i\) representa o valor reconstruído pelo autoencoder;
- \(N\) representa a quantidade de valores comparados.

No código do projeto, essa perda é calculada com `torch.nn.functional.mse_loss`. O otimizador Adam atualiza os pesos da rede para reduzir o erro de reconstrução ao longo das épocas de treinamento.

O modelo também acompanha a perda calculada no conjunto de validação. A implementação guarda os pesos associados ao melhor resultado de validação e utiliza *early stopping* para interromper o treinamento caso não haja melhora durante o número de épocas definido pela configuração.

### Por que utilizar o autoencoder neste projeto?

O autoencoder funciona como uma etapa de extração de características. Em vez de utilizar diretamente todas as variáveis pré-processadas para construir os perfis tabulares, o projeto aprende uma representação compacta dessas variáveis.

Essa representação é utilizada posteriormente pelo algoritmo K-Means, responsável por agrupar registros semelhantes e gerar a variável `tabular_profile_cluster`.

Portanto, o autoencoder **não classifica diretamente os chamados nas categorias Software, Rede, Hardware, RH, Acesso ou Financeiro**. Sua função é aprender uma representação dos dados estruturados que será utilizada na construção dos perfis tabulares e, posteriormente, como uma das entradas do classificador multimodal.


## 5) Treinamento do autoencoder

O treinamento é a etapa em que a rede neural ajusta seus parâmetros para aprender a reconstruir os dados de entrada com o menor erro possível. No Projeto Integrador II, esse processo é implementado na classe `TabularProfiles`, localizada em `src/models/profiles.py`.

O treinamento não utiliza diretamente os dados brutos. Antes de alimentar a rede neural, o projeto prepara as variáveis numéricas e categóricas para que possam ser processadas pelo PyTorch.

### Pré-processamento dos dados

A implementação aplica transformações diferentes de acordo com o tipo de variável.

**Variáveis numéricas:** valores ausentes são preenchidos utilizando a mediana calculada no conjunto de treinamento. Em seguida, as variáveis são padronizadas com `StandardScaler`, colocando-as em uma escala comparável.

**Variáveis categóricas:** os valores ausentes são representados pelo marcador `__missing__`, e as categorias são convertidas em colunas numéricas utilizando `OneHotEncoder`. Essa transformação permite representar categorias, como sistemas operacionais e tipos de dispositivo, em um formato que a rede neural consegue processar.

O projeto ajusta o pré-processador somente com os dados de treinamento. Depois, utiliza o mesmo pré-processador para transformar os dados de validação, evitando que informações estatísticas desse conjunto sejam utilizadas indevidamente durante o ajuste inicial.

Ao final, os dados transformados são convertidos para tensores do PyTorch com tipo `float32`, formato utilizado como entrada da rede neural.

### Separação entre treinamento e validação

O conjunto de treinamento é utilizado para atualizar os pesos do autoencoder. Já o conjunto de validação permite acompanhar o comportamento do modelo em registros que não participam dessas atualizações.

Essa separação é importante porque uma redução do erro nos dados de treinamento, isoladamente, não garante que o modelo esteja aprendendo uma representação que generalize bem para novos registros.

Na implementação, o treinamento acompanha duas métricas:

- `loss`: erro de reconstrução calculado nos dados de treinamento.
- `val_loss`: erro de reconstrução calculado nos dados de validação.

A comparação entre essas métricas ajuda a acompanhar a evolução do aprendizado ao longo das épocas.

### Ciclo de treinamento

O treinamento é realizado em lotes (*batches*), em vez de processar todos os registros de uma só vez. A configuração do projeto define um tamanho de lote de 64 registros e um máximo de 30 épocas.

Em cada lote, o processo segue estas etapas:

1. Os dados são enviados para o autoencoder.
2. O encoder gera a representação latente.
3. O decoder reconstrói os dados de entrada.
4. A função MSE calcula o erro entre os dados originais e os reconstruídos.
5. O método `zero_grad()` limpa os gradientes anteriores.
6. O método `backward()` calcula os gradientes necessários para ajustar os parâmetros.
7. O método `step()` do otimizador Adam atualiza os pesos da rede.

Esse ciclo é repetido para os lotes de treinamento de cada época. Ao final da época, o modelo calcula a perda de validação sem atualizar os pesos com esses dados.

### Otimizador e hiperparâmetros

Os principais parâmetros definidos na configuração `src/config/models.yaml` são:

| Parâmetro | Valor configurado | Finalidade |
|---|---:|---|
| `hidden_dim` | 32 | Dimensão da camada oculta |
| `latent_dim` | 8 | Dimensão da representação latente |
| `epochs` | 30 | Número máximo de épocas |
| `batch_size` | 64 | Quantidade de registros por lote |
| `learning_rate` | 0,001 | Taxa de aprendizado do otimizador Adam |
| `patience` | 5 | Épocas sem melhora toleradas antes da parada antecipada |

A taxa de aprendizado controla a intensidade dos ajustes realizados pelo otimizador a cada atualização. Uma taxa muito alta pode dificultar a convergência, enquanto uma taxa muito baixa pode tornar o aprendizado mais lento.

Esses valores representam os parâmetros definidos no projeto. O número efetivo de épocas executadas depende do comportamento da perda de validação e da aplicação do critério de parada antecipada.

### Early stopping e recuperação do melhor modelo

O projeto utiliza *early stopping* para evitar que o treinamento continue indefinidamente sem apresentar melhora na validação.

Sempre que a perda de validação melhora, os pesos correspondentes são guardados. Caso não haja melhora durante cinco épocas consecutivas, conforme a configuração, o treinamento pode ser interrompido antes de atingir o limite máximo.

Ao final, a implementação restaura os pesos associados à melhor perda de validação observada, em vez de necessariamente manter os pesos da última época executada.

Essa estratégia permite selecionar o estado do modelo que apresentou o melhor resultado de reconstrução no conjunto de validação.

### Resultado do treinamento

Ao concluir essa etapa, o autoencoder possui pesos ajustados para reconstruir as variáveis pré-processadas e gerar representações latentes por meio do encoder.

O código registra o histórico das perdas de treinamento e validação, além de informações como a quantidade de registros utilizados e a melhor perda de validação observada.

Esses resultados servem como base para a próxima etapa: utilizar as representações latentes para agrupar registros semelhantes com o algoritmo K-Means.

É importante distinguir o objetivo dessa etapa do objetivo do classificador final. O autoencoder é treinado para reconstruir os dados, e não para minimizar diretamente os erros na previsão da categoria do chamado.

## 6) Geração de perfis tabulares com K-Means

Após o treinamento do autoencoder, o projeto utiliza as representações latentes produzidas pelo encoder para identificar grupos de registros com características semelhantes. Essa etapa é implementada em `src/models/profiles.py` e utiliza o algoritmo K-Means.

O K-Means é um algoritmo de aprendizado não supervisionado que organiza os registros em uma quantidade predefinida de grupos, chamados de *clusters*. Para isso, utiliza a distância entre as representações numéricas dos registros, buscando formar grupos internamente semelhantes.

Neste projeto, o K-Means não trabalha diretamente com todas as variáveis originais. Ele recebe a representação compacta de oito dimensões produzida pelo encoder do autoencoder.

### Da representação latente aos clusters

O processo de geração dos perfis segue estas etapas:

1. Os dados passam pelo pré-processador já ajustado durante o treinamento.
2. O encoder transforma os dados em representações latentes de oito dimensões.
3. Um `StandardScaler` padroniza essas representações latentes.
4. O K-Means agrupa os registros a partir das representações padronizadas.
5. Cada registro recebe um identificador correspondente ao cluster atribuído.

A padronização da representação latente é realizada antes do agrupamento para colocar suas dimensões em escalas comparáveis durante o cálculo das distâncias.

### Configuração do K-Means

O projeto define seis clusters por meio da configuração `n_clusters: 6`, presente em `src/config/models.yaml`.

Na implementação, o K-Means utiliza `random_state=seed` e `n_init=20`. O primeiro parâmetro controla a aleatoriedade associada à inicialização, enquanto o segundo determina quantas inicializações diferentes são realizadas para buscar uma solução de agrupamento.

O algoritmo procura posicionar os centroides de forma a minimizar a soma das distâncias quadráticas entre os registros e os centroides dos grupos aos quais foram atribuídos.

É importante observar que a definição de seis clusters é uma escolha de configuração do projeto. Ela não significa que existam exatamente seis tipos naturais de chamados nem que cada cluster corresponda a uma das seis categorias de classificação.

### O que representa `tabular_profile_cluster`?

Após o agrupamento, o projeto gera a variável `tabular_profile_cluster`, que identifica o grupo atribuído a cada registro.

Por exemplo, um registro pode receber o identificador `0`, enquanto outro pode receber o identificador `3`. Esses valores representam apenas os clusters atribuídos pelo algoritmo, e não categorias como Software, Rede ou Hardware.

Os números dos clusters também não representam uma escala de prioridade. O cluster `5`, por exemplo, não é necessariamente mais urgente do que o cluster `1`.

Como o K-Means utiliza distâncias entre representações numéricas, os grupos refletem os padrões encontrados pelo algoritmo nas características aprendidas pelo autoencoder. A interpretação prática de cada grupo depende de uma análise posterior dos registros que o compõem.

### Por que os perfis são úteis para o classificador multimodal?

O perfil tabular funciona como uma característica adicional que resume padrões presentes nas variáveis estruturadas dos chamados.

Na configuração do classificador multimodal, `tabular_profile_cluster` está incluída entre as variáveis categóricas. Dessa forma, o modelo final pode utilizar essa informação junto com outras características numéricas, textuais e relacionadas às imagens.

O fluxo pode ser resumido da seguinte forma:

**Variáveis estruturadas → Pré-processamento → Autoencoder → Representação latente → K-Means → `tabular_profile_cluster` → Classificador multimodal**

Essa abordagem permite incorporar uma informação derivada do agrupamento não supervisionado ao processo de classificação supervisionada.

Entretanto, a criação dos perfis não garante, por si só, uma melhora na acurácia do classificador. Para verificar se essa característica realmente contribui para o desempenho, seria necessário comparar os resultados do modelo com e sem a variável `tabular_profile_cluster`, utilizando o mesmo protocolo de avaliação.

### Transformação de novos registros

Depois de ajustado, o pipeline reutiliza o pré-processador, o encoder, o `StandardScaler` da representação latente e o K-Means já treinados para processar novos registros.

Isso significa que os novos dados são transformados usando os parâmetros aprendidos anteriormente, sem reajustar os componentes durante a etapa de transformação.

Essa separação é importante para manter a consistência entre os perfis gerados durante o treinamento e aqueles atribuídos posteriormente. O método `transform()` retorna a variável `tabular_profile_cluster` como uma categoria, pronta para ser integrada ao pipeline multimodal.

Em resumo, o K-Means transforma a representação latente do autoencoder em uma variável categórica que resume o agrupamento dos registros. Essa variável passa a integrar as características utilizadas pelo classificador final, mas não substitui a categoria real do chamado nem realiza a classificação supervisionada por conta própria.


## 7) Integração do Deep Learning com o pipeline multimodal

O objetivo do Projeto Integrador II é classificar chamados de suporte de TI em seis categorias: Software, Rede, Hardware, RH, Acesso e Financeiro. Para isso, o projeto combina informações estruturadas, textuais e visuais em um único pipeline de classificação.

O Deep Learning participa desse processo de duas maneiras distintas: primeiro, por meio do autoencoder, que aprende representações das variáveis estruturadas e auxilia na geração dos perfis tabulares; depois, por meio do modelo multimodal, que utiliza as características combinadas para prever a categoria do chamado.

### As diferentes fontes de informação

O pipeline utiliza características provenientes de diferentes fontes:

**Dados estruturados:** incluem variáveis como `sla_hours`, `sentiment_score`, `num_words`, `device_type`, `os` e `network_type`. O autoencoder utiliza parte dessas informações para aprender representações latentes e gerar os perfis tabulares.

**Dados textuais:** incluem o título e a descrição do chamado. O pipeline de processamento de linguagem natural (PLN) extrai características que ajudam a representar o conteúdo da solicitação, como o tópico principal e o sentimento identificado.

**Dados visuais:** são utilizados para extrair características relacionadas às imagens dos chamados. A configuração prevê informações como `image_score`, enquanto o pipeline de visão computacional pode produzir descrições das imagens.

Essas fontes oferecem perspectivas complementares sobre um mesmo chamado. Por exemplo, a descrição pode indicar um problema de conexão, as variáveis estruturadas podem informar o tipo de rede e uma imagem pode apresentar evidências visuais relacionadas ao problema.

Esse exemplo é apenas ilustrativo: a contribuição efetiva de cada fonte precisa ser avaliada experimentalmente.

### O papel do autoencoder na integração

O autoencoder não combina diretamente todas as modalidades. Sua responsabilidade é trabalhar com as variáveis tabulares selecionadas, aprender uma representação latente e gerar os agrupamentos utilizados para criar a variável `tabular_profile_cluster`.

Essa variável é adicionada às características categóricas utilizadas pelo classificador multimodal.

Assim, o resultado do Deep Learning tabular passa a fazer parte do conjunto de informações disponíveis para o modelo final, juntamente com as características extraídas do texto, das imagens e das demais variáveis estruturadas.

### Fusão das características

A etapa de fusão reúne as características selecionadas para construir a entrada do classificador multimodal. No projeto, essa integração é configurada em `src/config/models.yaml` e implementada nos componentes do pipeline multimodal, incluindo `src/models/fusion.py` e `src/models/fusion_pipeline.py`.

A configuração prevê três grupos principais de características:

- **Numéricas:** `sla_hours`, `sentiment_score`, `num_words`, `image_score`, `semantic_margin` e `urgency_score`.
- **Categóricas:** `device_type`, `os`, `network_type`, `main_topic` e `tabular_profile_cluster`.
- **Textuais e visuais:** o título e a descrição do chamado, além da descrição associada à imagem.

A combinação dessas informações permite que o classificador considere diferentes aspectos do chamado durante a previsão. As variáveis numéricas e categóricas são processadas conforme seus tipos, enquanto os campos textuais e visuais participam da representação definida pelo pipeline.

A fusão não significa simplesmente concatenar os textos originais, as imagens e os números sem tratamento. Cada modalidade precisa ser transformada para uma representação compatível com o modelo utilizado na classificação.

### O classificador multimodal

Diferentemente do autoencoder, o classificador multimodal realiza aprendizado supervisionado: durante o treinamento, utiliza as características de entrada e as categorias conhecidas dos chamados para aprender a prever a variável-alvo `target_category`.

A configuração do projeto contempla a comparação entre uma regressão logística e redes neurais do tipo MLP (*Multilayer Perceptron*). Para a MLP, são definidos candidatos de arquitetura com camadas ocultas de 64 neurônios ou de 128 e 64 neurônios, além de diferentes valores de dropout, taxa de aprendizado, regularização e tamanho de lote.

Esses candidatos fazem parte da configuração de busca de hiperparâmetros. Sua presença não significa que uma arquitetura específica tenha sido comprovadamente superior, pois isso depende da execução do treinamento e da avaliação dos resultados.

O objetivo final é produzir uma previsão para uma das seis categorias definidas no projeto.

### Fluxo completo do pipeline

O processo pode ser resumido em cinco etapas:

1. **Preparação dos dados:** as informações dos chamados são organizadas e as variáveis necessárias são preparadas para processamento.
2. **Extração de características:** os pipelines de visão computacional, PLN e Deep Learning tabular produzem características a partir de suas respectivas fontes.
3. **Geração de perfis tabulares:** o encoder produz representações latentes, e o K-Means atribui os registros a grupos, gerando `tabular_profile_cluster`.
4. **Fusão multimodal:** as características numéricas, categóricas, textuais e visuais são reunidas conforme a configuração do modelo.
5. **Classificação supervisionada:** o modelo final utiliza as características combinadas para prever `target_category`.

### Como avaliar a contribuição do Deep Learning

A utilização de uma arquitetura mais complexa não garante, por si só, melhores resultados. Para entender a contribuição do autoencoder e dos perfis tabulares, é importante comparar modelos sob as mesmas condições de treinamento e avaliação.

Uma possibilidade é comparar o classificador multimodal completo com uma versão que não utilize `tabular_profile_cluster`. Essa comparação permite investigar se a informação gerada pelo autoencoder e pelo K-Means contribui para o desempenho da classificação.

Também é importante manter a separação adequada entre treinamento, validação e teste, evitando que informações dos conjuntos de avaliação influenciem o ajuste dos componentes do pipeline.

Dessa forma, o Deep Learning tabular é tratado como uma etapa de engenharia de características integrada ao sistema multimodal, enquanto a qualidade da classificação final deve ser determinada pelos resultados experimentais.

## 8) Prevenção de vazamento de dados e cuidados na avaliação

Durante o desenvolvimento de modelos de Machine Learning e Deep Learning, é necessário garantir que as informações utilizadas no treinamento não comprometam a avaliação do modelo. Um dos principais riscos nesse processo é o *data leakage*, ou vazamento de dados.

O vazamento acontece quando informações que deveriam estar restritas à validação ou ao teste acabam influenciando o treinamento ou o ajuste dos componentes do pipeline. Isso pode produzir métricas excessivamente otimistas, que não representam adequadamente o comportamento do modelo diante de novos chamados.

No Projeto Integrador II, esse cuidado é especialmente relevante porque o pipeline inclui pré-processamento, treinamento do autoencoder, geração de perfis com K-Means e classificação multimodal.

### Ajuste do pré-processamento

O pré-processamento utiliza parâmetros aprendidos a partir dos dados, como medianas para preencher valores ausentes e médias e desvios para padronizar variáveis numéricas.

Por isso, esses parâmetros devem ser ajustados somente com o conjunto de treinamento. Os conjuntos de validação e teste devem receber as mesmas transformações, sem participar do cálculo dos parâmetros.

A implementação de `TabularProfiles.fit()` segue essa lógica ao ajustar o pré-processador com os dados de treinamento e utilizá-lo posteriormente para transformar os dados de validação.

### Treinamento do autoencoder

O autoencoder aprende seus pesos durante o treinamento. Se registros do conjunto de teste forem utilizados para ajustar a rede, mesmo sem fornecer suas categorias, as informações desses registros poderão influenciar a representação aprendida.

Por esse motivo, o treinamento do autoencoder deve utilizar apenas os dados permitidos para essa etapa. A validação pode ser utilizada para acompanhar a perda e selecionar os melhores pesos, conforme implementado no projeto.

O conjunto de teste deve permanecer separado até a avaliação final.

### Padronização latente e K-Means

Após o treinamento do autoencoder, o projeto ajusta um `StandardScaler` sobre as representações latentes do conjunto de treinamento e utiliza essas representações para ajustar o K-Means.

Essa ordem é importante porque tanto a padronização quanto os centroides do K-Means dependem dos dados utilizados no ajuste.

Para processar novos registros, o pipeline deve reutilizar o encoder, o scaler latente e o agrupador já ajustados. Não se deve recalcular esses componentes com base no conjunto de teste durante a avaliação, pois isso alteraria a representação e os agrupamentos a partir dos próprios dados avaliados.

### Separação entre as categorias e as características

O autoencoder tabular do projeto é treinado para reconstruir as variáveis de entrada, sem utilizar diretamente `target_category` como objetivo de treinamento. Essa característica o diferencia do classificador multimodal, que aprende a prever a categoria correta dos chamados.

Entretanto, o fato de o autoencoder não utilizar o alvo não elimina automaticamente todos os riscos de vazamento. É necessário verificar também como os dados são separados, como as características são geradas e se algum componente utiliza informações que não estariam disponíveis no momento de uma previsão real.

Além disso, a variável `tabular_profile_cluster` representa um agrupamento não supervisionado, e não uma codificação da categoria correta do chamado.

### Avaliação justa do classificador multimodal

Para avaliar adequadamente o classificador, é necessário manter o conjunto de teste independente das decisões de treinamento e seleção de hiperparâmetros.

A validação pode ser utilizada para comparar configurações e escolher o modelo. O teste deve ser reservado para estimar o desempenho da configuração selecionada.

Também é importante que as comparações entre versões do modelo utilizem as mesmas divisões de dados e critérios de avaliação. Assim, é possível investigar se a inclusão dos perfis tabulares realmente contribui para a classificação, sem confundir melhorias aparentes com diferenças no protocolo experimental.

Em problemas de chamados, outro cuidado é verificar se existem registros duplicados ou chamados muito relacionados distribuídos entre diferentes conjuntos. Dependendo da estrutura dos dados, uma divisão inadequada pode permitir que informações semelhantes apareçam tanto no treinamento quanto no teste.

### Persistência dos componentes treinados

O projeto prevê o armazenamento dos pesos do autoencoder em `autoencoder.pt`, dos pré-processadores e do agrupador em `preprocessors.joblib`, além de informações sobre a arquitetura e o histórico de treinamento em `network.json`.

A persistência permite reutilizar os componentes ajustados sem precisar treiná-los novamente a cada transformação. Também contribui para manter a consistência entre a geração dos perfis tabulares e sua utilização posterior no pipeline multimodal.

É importante que os artefatos utilizados na avaliação correspondam à versão efetivamente treinada e selecionada, mantendo a rastreabilidade do processo.

### Síntese dos cuidados necessários

Os principais cuidados são:

- Ajustar os pré-processadores somente com os dados de treinamento.
- Treinar o autoencoder sem utilizar o conjunto de teste.
- Ajustar o scaler latente e o K-Means somente com as representações permitidas para treinamento.
- Reutilizar os componentes ajustados ao transformar dados de validação e teste.
- Utilizar a validação para selecionar modelos e hiperparâmetros, preservando o teste para a avaliação final.
- Investigar duplicatas e possíveis relações entre registros que possam comprometer a separação dos conjuntos.
- Não interpretar a ausência de uso direto de `target_category` no autoencoder como garantia absoluta contra vazamento.

Esses cuidados ajudam a tornar a avaliação mais confiável e permitem analisar o desempenho do classificador multimodal de maneira mais próxima das condições em que ele seria utilizado na prática.


## 9) Persistência, limitações e considerações finais

### Persistência do modelo

Após o treinamento, os componentes aprendidos pelo modelo precisam ser armazenados para que possam ser reutilizados posteriormente, sem a necessidade de treinar o autoencoder e os demais componentes novamente a cada execução.

No arquivo `src/models/profiles.py`, a implementação de `TabularProfiles` prevê o armazenamento dos seguintes artefatos:

| Arquivo | Finalidade |
|---|---|
| `autoencoder.pt` | Armazena os pesos aprendidos pela rede neural do autoencoder. |
| `preprocessors.joblib` | Armazena os componentes de pré-processamento, o escalonador da representação latente e o modelo K-Means. |
| `network.json` | Registra informações sobre a arquitetura da rede e o histórico de treinamento. |

Essa separação permite preservar tanto os parâmetros aprendidos pela rede neural quanto os componentes necessários para transformar os dados e gerar os perfis tabulares.

Na etapa de utilização, o método de restauração recupera esses componentes. Assim, novos chamados podem passar pelo mesmo pré-processamento e pela mesma transformação da representação latente, utilizando o agrupamento já ajustado durante o treinamento.

É importante que os dados novos sejam processados com os componentes previamente ajustados. Reajustar o pré-processador ou o K-Means a cada lote poderia alterar a representação dos dados e o significado operacional dos perfis gerados.

### Limitações e cuidados na utilização

Apesar de contribuir para a engenharia de atributos, a arquitetura possui limitações que precisam ser consideradas.

**Dependência da qualidade dos dados:** o autoencoder aprende a reconstruir os dados que recebe. Valores ausentes, inconsistências, variáveis pouco informativas ou dados que não representem adequadamente os chamados podem prejudicar a qualidade da representação aprendida.

**Interpretação dos perfis:** o K-Means gera seis agrupamentos conforme a configuração atual. Entretanto, não existe garantia de que esses grupos correspondam a categorias de negócio bem definidas. A interpretação de cada perfil depende da análise das características dos chamados agrupados.

**Ausência de classificação direta:** o autoencoder é treinado para reconstruir as próprias entradas, utilizando a função de perda MSE (*Mean Squared Error*). Ele não aprende diretamente a prever as categorias Software, Rede, Hardware, RH, Acesso e Financeiro. Essa tarefa pertence ao classificador supervisionado do pipeline multimodal.

**Dependência da configuração:** parâmetros como as dimensões das camadas, a taxa de aprendizado, o número de épocas e a quantidade de agrupamentos influenciam o comportamento do processo. Os valores definidos no arquivo `src/config/models.yaml` representam a configuração do projeto, mas não demonstram, por si só, que sejam os melhores parâmetros possíveis.

**Necessidade de avaliação experimental:** a inclusão de `tabular_profile_cluster` pode ou não melhorar a classificação final. Para verificar sua contribuição, seria necessário comparar experimentos com e sem essa variável, mantendo os mesmos dados, divisões e critérios de avaliação.

**Generalização para novos dados:** o modelo precisa ser avaliado com chamados que não participaram do treinamento. Mudanças no perfil dos chamados ao longo do tempo também podem exigir monitoramento e, eventualmente, um novo treinamento.

### O papel do Deep Learning no projeto

Neste projeto, o Deep Learning atua como uma etapa de engenharia de atributos para os dados tabulares. O autoencoder transforma as variáveis estruturadas em uma representação latente compacta, e o K-Means utiliza essa representação para identificar agrupamentos de chamados.

O identificador do agrupamento é então disponibilizado como a variável categórica `tabular_profile_cluster`, que pode ser utilizada pelo classificador multimodal junto às informações textuais, visuais e estruturadas.

Dessa forma, o fluxo de processamento pode ser resumido em cinco etapas:

1. **Preparação dos dados:** seleção das variáveis e aplicação do pré-processamento.
2. **Aprendizado da representação:** treinamento do autoencoder para reconstruir os dados tabulares.
3. **Geração dos perfis:** aplicação do K-Means sobre a representação latente.
4. **Integração multimodal:** combinação dos perfis tabulares com atributos extraídos de textos, imagens e outras variáveis.
5. **Classificação supervisionada:** utilização dos atributos combinados para prever a categoria do chamado.

O ponto principal é que o autoencoder não substitui o classificador final. Sua função é produzir uma representação adicional dos dados, que poderá contribuir para a decisão do modelo multimodal.

### Considerações finais

A implementação demonstra como uma rede neural pode ser utilizada além da classificação tradicional, atuando também na extração de representações e na geração de atributos para outras etapas de um pipeline de Machine Learning.

Ao combinar um autoencoder com o K-Means, o projeto incorpora uma etapa não supervisionada de geração de perfis tabulares. Essa informação pode complementar os atributos provenientes do processamento de linguagem natural e da visão computacional, ampliando o conjunto de informações disponível para o classificador multimodal.

Entretanto, a existência dessa arquitetura não comprova, isoladamente, uma melhoria na classificação dos chamados. A contribuição efetiva do Deep Learning deverá ser verificada por meio de experimentos controlados, métricas adequadas e comparação com alternativas mais simples.

Portanto, o valor dessa abordagem está na integração entre preparação de dados, aprendizado de representações, agrupamento e classificação supervisionada, mantendo cada componente responsável por uma função específica dentro do sistema.